# Benchmark de transcripción en GPU — Tarea 5

Este notebook corre en **Google Colab con GPU** y mide lo que pide la fase 2 de la Tarea 1:
dos tamaños de modelo, GPU contra CPU, float16 contra int8, glosario y **VRAM medida**.

**Antes de empezar:**
1. `Entorno de ejecución → Cambiar tipo de entorno de ejecución → T4 GPU`.
2. Sube tus 10 audios (`E01.m4a` … `V05.m4a`) a una carpeta de Google Drive llamada **`Tarea5_audio`**.
3. `Entorno de ejecución → Ejecutar todo`. Tarda unos 20–30 minutos.

In [1]:
# 1. Evidencia de hardware: qué GPU asignó Colab
!nvidia-smi

Sat Oct 31 01:14:19 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   43C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
# 2. Código del proyecto (repositorio público)
REPO = "https://github.com/henryluis0603/Tarea-5-Data-Science-con-Python.git"
RAMA = "feat/evaluacion-gpu"   # cuando los PRs estén fusionados, también sirve "main"
!rm -rf /content/proyecto && git clone -q -b $RAMA $REPO /content/proyecto
%cd /content/proyecto
!pip install -q -r requirements-colab.txt

/content/proyecto


In [3]:
# 2b. CTranslate2 (motor de faster-whisper) necesita cuBLAS y cuDNN de CUDA 12.
#     Colab ahora trae CUDA 13, así que se instalan como paquetes de pip y se exponen
#     con LD_LIBRARY_PATH (los comandos con ! heredan esta variable).
!pip install -q nvidia-cublas-cu12 "nvidia-cudnn-cu12==9.*"
import os, nvidia.cublas.lib, nvidia.cudnn.lib
libs = [list(nvidia.cublas.lib.__path__)[0], list(nvidia.cudnn.lib.__path__)[0]]
os.environ["LD_LIBRARY_PATH"] = ":".join(libs + [os.environ.get("LD_LIBRARY_PATH", "")])
print(os.environ["LD_LIBRARY_PATH"])
!ls $LD_LIBRARY_PATH | grep -E "libcublas.so.12|libcudnn.so.9" | head

/usr/local/lib/python3.13/dist-packages/nvidia/cublas/lib:/usr/local/lib/python3.13/dist-packages/nvidia/cudnn/lib:/usr/lib64-nvidia
ls: cannot access '/usr/local/lib/python3.13/dist-packages/nvidia/cublas/lib:/usr/local/lib/python3.13/dist-packages/nvidia/cudnn/lib:/usr/lib64-nvidia': No such file or directory


In [4]:
# 3. Audios desde Google Drive (no están en el repositorio)
from google.colab import drive
drive.mount('/content/drive')
!mkdir -p audio && cp /content/drive/MyDrive/Tarea5_audio/*.m4a audio/ && ls -la audio

Mounted at /content/drive
total 12392
drwxr-xr-x 2 root root    4096 Oct 31 01:15 .
drwxr-xr-x 9 root root    4096 Oct 31 01:14 ..
-rw------- 1 root root 1635008 Oct 31 01:15 E01.m4a
-rw------- 1 root root 1323591 Oct 31 01:15 E02.m4a
-rw------- 1 root root 1322856 Oct 31 01:15 E03.m4a
-rw------- 1 root root 1271005 Oct 31 01:15 E04.m4a
-rw------- 1 root root 1312655 Oct 31 01:15 E05.m4a
-rw-r--r-- 1 root root     210 Oct 31 01:14 LEEME.md
-rw------- 1 root root 1318009 Oct 31 01:15 V01.m4a
-rw------- 1 root root 1154280 Oct 31 01:15 V02.m4a
-rw------- 1 root root 1117549 Oct 31 01:15 V03.m4a
-rw------- 1 root root 1120266 Oct 31 01:15 V04.m4a
-rw------- 1 root root 1081570 Oct 31 01:15 V05.m4a


In [5]:
# 4. El dispositivo se lee de la librería en tiempo de ejecución, no de config.yaml
import ctranslate2, faster_whisper
print("faster-whisper", faster_whisper.__version__, "| ctranslate2", ctranslate2.__version__)
print("GPUs CUDA visibles para CTranslate2:", ctranslate2.get_cuda_device_count())

faster-whisper 1.2.1 | ctranslate2 4.8.4
GPUs CUDA visibles para CTranslate2: 1


In [6]:
# 4b. Prueba rápida: si esto falla, el benchmark también fallaría
!python -c "from faster_whisper import WhisperModel; m=WhisperModel('tiny', device='cuda', compute_type='float16'); s,_=m.transcribe('audio/E01.m4a', language='es'); print('GPU OK:', next(iter(s)).text, '| device:', m.model.device)"

GPU OK:  Buenas tardes, Dani, eres por conectarte, soy del área de selección de datos andines. ¿Cómo estás? | device: cuda


In [7]:
# 5. Benchmark completo (modelos x precisiones x dispositivo + glosario)
!python -m scripts.benchmark_gpu --clips-cpu E01 E02

/content/proyecto/scripts/benchmark_gpu.py:30: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml for you.
  import pynvml
small            cuda  float16  ninguno         E01 WER=0.077 1.8s/min dev=cuda VRAM=1302.1875
small            cuda  float16  ninguno         E02 WER=0.074 1.8s/min dev=cuda VRAM=1334.1875
small            cuda  float16  ninguno         E03 WER=0.089 1.6s/min dev=cuda VRAM=1334.1875
small            cuda  float16  ninguno         E04 WER=0.074 1.8s/min dev=cuda VRAM=1334.1875
small            cuda  float16  ninguno         E05 WER=0.073 1.6s/min dev=cuda VRAM=1334.1875
small            cuda  int8     ninguno         E01 WER=0.075 1.7s/min dev=cuda VRAM=1046.1875
small            cuda  int8     ninguno         E02 WER=0.090 1.7s/min dev=cuda VRAM=1046.1875
small            cuda  int8     ninguno         E03 WER=0.089 

In [8]:
# 6. Resumen
import json, pandas as pd
r = json.load(open("results/benchmark_gpu.json"))
df = pd.DataFrame(r["filas"]).drop(columns=["texto"])
print("GPU:", r["gpu"])
display(df.groupby(["modelo","dispositivo_pedido","dispositivo_real","precision","modo_glosario"])
          [["wer","cer","seg_por_min_audio","vram_pico_mb","vram_modelo_mb"]].mean().round(3))
print("¿Cambia el texto float16 vs int8?"); display(r["precision_cambia_texto"])
print("¿Cambia el texto GPU vs CPU?"); display(r["cpu_vs_gpu_cambia_texto"])

GPU: Tesla T4, 15360 MiB, 580.82.07


wer  \
modelo         dispositivo_pedido dispositivo_real precision modo_glosario           
large-v3-turbo cpu                cpu              int8      ninguno         0.040   
               cuda               cuda             float16   hotwords        0.250   
                                                             initial_prompt  0.048   
                                                             ninguno         0.045   
                                                   int8      ninguno         0.052   
small          cuda               cuda             float16   ninguno         0.077   
                                                   int8      ninguno         0.079   

                                                                               cer  \
modelo         dispositivo_pedido dispositivo_real precision modo_glosario           
large-v3-turbo cpu                cpu              int8      ninguno         0.018   
               cuda               cuda             float16   hotwords        0.207   
                                                             initial_prompt  0.020   
                                                             ninguno         0.019   
                                                   int8      ninguno         0.022   
small          cuda               cuda             float16   ninguno         0.031   
                                                   int8      ninguno         0.035   

                                                                             seg_por_min_audio  \
modelo         dispositivo_pedido dispositivo_real precision modo_glosario                       
large-v3-turbo cpu                cpu              int8      ninguno                    70.130   
               cuda               cuda             float16   hotwords                    2.758   
                                                             initial_prompt              1.978   
                                                             ninguno                     1.854   
                                                   int8      ninguno                     1.808   
small          cuda               cuda             float16   ninguno                     1.694   
                                                   int8      ninguno                     1.706   

                                                                             vram_pico_mb  \
modelo         dispositivo_pedido dispositivo_real precision modo_glosario                  
large-v3-turbo cpu                cpu              int8      ninguno              556.188   
               cuda               cuda             float16   hotwords            2524.588   
                                                             initial_prompt      2390.188   
                                                             ninguno             2390.188   
                                                   int8      ninguno             1590.188   
small          cuda               cuda             float16   ninguno             1327.788   
                                                   int8      ninguno             1046.188   

                                                                             vram_modelo_mb  
modelo         dispositivo_pedido dispositivo_real precision modo_glosario                   
large-v3-turbo cpu                cpu              int8      ninguno                    0.0  
               cuda               cuda             float16   hotwords                1968.4  
                                                             initial_prompt          1834.0  
                                                             ninguno                 1834.0  
                                                   int8      ninguno                 1034.0  
small          cuda               cuda             float16   ninguno                  877.6  
                                                   int8 

¿Cambia el texto float16 vs int8?


{'small': [{'clip': 'E01',
   'texto_identico': False,
   'wer_entre_si': 0.009237875288683603},
  {'clip': 'E02',
   'texto_identico': False,
   'wer_entre_si': 0.046153846153846156},
  {'clip': 'E03',
   'texto_identico': False,
   'wer_entre_si': 0.020527859237536656},
  {'clip': 'E04',
   'texto_identico': False,
   'wer_entre_si': 0.024861878453038673},
  {'clip': 'E05',
   'texto_identico': False,
   'wer_entre_si': 0.014792899408284023}],
 'large-v3-turbo': [{'clip': 'E01',
   'texto_identico': False,
   'wer_entre_si': 0.004608294930875576},
  {'clip': 'E02', 'texto_identico': True, 'wer_entre_si': 0.0},
  {'clip': 'E03',
   'texto_identico': False,
   'wer_entre_si': 0.011627906976744186},
  {'clip': 'E04',
   'texto_identico': False,
   'wer_entre_si': 0.06925207756232687},
  {'clip': 'E05',
   'texto_identico': False,
   'wer_entre_si': 0.011730205278592375}]}

¿Cambia el texto GPU vs CPU?


[{'clip': 'E01', 'texto_identico': True, 'wer_entre_si': 0.0},
 {'clip': 'E02',
  'texto_identico': False,
  'wer_entre_si': 0.0076726342710997444}]

In [9]:
# 7. Descarga los resultados y ponlos en la carpeta del proyecto en tu computadora
!zip -qr resultados_gpu.zip results/benchmark_gpu.json data/transcripciones/benchmark
from google.colab import files
files.download("resultados_gpu.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>